# Topic 1: Throughput vs. Latency (The Core Dilemma)

Welcome back! Now that we have verified our environment and established a timing ritual, we need to dive into the physical substrate. Why do CPUs and GPUs exist as separate architectures? It comes down to a fundamental engineering tradeoff: **Latency vs. Throughput**.

### What We Will Learn
* **Latency vs. Throughput:** Understand the architectural design choices of CPUs and GPUs.
* **Sequential vs. Parallel Math:** Observe how CPU sequential execution compares to GPU parallel execution.
* **Hardware Mapping:** Visualize the difference between massive complex cores and thousands of simple ALUs.

### The Backdrop
A CPU is designed to execute a single thread of execution as fast as possible. It is a **latency monster**. To do this, it devotes massive space to cache memory (to avoid waiting for RAM) and control logic (for branch prediction and out-of-order execution). 

A GPU, on the other hand, is a **throughput monster**. It doesn't care about making a single thread run fast; it wants to execute millions of threads in parallel. To do this, it throws away branch prediction and huge caches, and fills that space with thousands of simple Arithmetic Logic Units (ALUs).

Let's see this in action.

### Visualizing the Architectures

Here is a visual sketch contrasting a latency-optimized CPU (with massive control logic and caches) and a throughput-optimized GPU (packed with tiny parallel execution units):

![CPU vs GPU Architecture](images/cpu-vs-gpu-architecture.svg)

### Step 1: Let's import our tools

We'll import PyTorch and time. We'll use these to test sequential loops on the CPU and parallel execution on the GPU.

In [ ]:
import time  # For wall-clock timing measurements
import torch  # PyTorch library for math operations

This imports standard timing utilities and PyTorch. Now, let's verify if our GPU backend is active and assign our target device.

### Step 2: Querying the device target

We check if CUDA is available and prepare our device string so we can place workloads appropriately.

In [ ]:
# Query CUDA availability and set active device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active device: {device}")

If it prints `cuda`, we can proceed to test both processors. Let's start with a CPU baseline.

### Step 3: Measuring Latency (The CPU Way)

We will simulate a sequential math task on the CPU: doing element-wise operations one by one. We'll create a 1D tensor with 1,000,000 floats and calculate their squares in a Python loop to emulate a single-threaded CPU workload.

In [ ]:
# Set up a sequential computation loop on the CPU
data = torch.randn(1000000, device="cpu")
start_time = time.perf_counter()  # Start CPU timer
for i in range(len(data)):
    data[i] = data[i] ** 2  # Square each element sequentially
print(f"CPU sequential: {time.perf_counter() - start_time:.4f}s")

This operation takes a noticeable fraction of a second. This is because the CPU core, although very fast, must step through the list one element at a time.

### Step 4: Measuring Throughput (The GPU Way)

Now let's run the exact same operation on the GPU. Instead of a loop, we write `data_gpu ** 2`. PyTorch compiles this into a vectorized CUDA kernel, launching thousands of parallel threads on the GPU's ALUs.

In [ ]:
# Run vectorized square operation on the GPU with sync
data_gpu = torch.randn(1000000, device=device)
if device == "cuda":
    torch.cuda.synchronize()  # Clear CUDA stream
start_time = time.perf_counter()  # Start CPU timer
result = data_gpu ** 2  # Square all 1,000,000 elements in parallel
if device == "cuda":
    torch.cuda.synchronize()  # Block until parallel GPU work is done
print(f"GPU parallel: {time.perf_counter() - start_time:.5f}s")

Look at that speedup. What took the CPU sequential loop several hundred milliseconds is calculated almost instantaneously on the GPU. This is because the GPU doesn't loop; it processes all one million operations in parallel using its massive grid of execution units.

## First-Principles Checkpoint: Throughput vs. Latency

Let's pause and make sure we have the core distinction clear:

1. **CPUs are Latency Engines:** They are designed to minimize the time it takes to complete a single instruction. If your code is full of branches (`if/else`) or runs sequentially, a CPU core is far superior because its clock speed is high and it can speculate branch paths.
2. **GPUs are Throughput Engines:** They are designed to maximize the total number of operations completed per second. If you have millions of independent operations (like squaring elements in an array or doing matrix multiplies), the GPU wins because it executes them simultaneously.

### Role Lens: Why it matters in practice
*   **DevOps / MLOps:** Knowing this tradeoff prevents you from throwing expensive GPU hardware at sequential workloads (like standard Python loops or web requests) where CPU cores would run circles around them.
*   **Data Science:** You realize why Deep Learning models must be formulated as tensor operations (like matrix multiplications) rather than loops. A model designed with loops will crawl, while one expressed as a tensor operation will fly.
*   **Data Engineering:** When choosing between pandas (CPU-bound) and Polars GPU/cuDF (GPU-bound), you select the GPU approach only when the dataset is large enough that parallel throughput overrides the startup latency of setting up the GPU.

In the next topic, we will look at the cost of moving data to the GPU in the first place—the famous PCIe bottleneck.